# 🚀 Hope ML: Professional Training Pipeline

This notebook provides a robust, professional environment for training the Hope Trading Transformer. Optimized for **Antigravity IDE** and **Google Colab**.

### Key Features:
*   **IDE Integration**: Secret management via GDrive `.env` file.
*   **Path Agnostic**: Auto-detects project location in Google Drive.
*   **Observability**: Real-time TensorBoard monitoring and data distribution plots.
*   **Integrity**: Automatic repo synchronization and dependency validation.

In [ ]:
# @title ## 1. Drive Connection & Path Setup
from google.colab import drive
import os
import sys
import subprocess

# 1. Mount Drive
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

# 2. Auto-detect Project Root
POSSIBLE_ROOTS = [
    "/content/drive/MyDrive/hope",
    "/content/drive/MyDrive/hope-trading",
    "/content/hope",
    os.getcwd()
]

PROJECT_ROOT = None
for root in POSSIBLE_ROOTS:
    if os.path.exists(os.path.join(root, "scripts/train_fixed.py")):
        PROJECT_ROOT = root
        break

if not PROJECT_ROOT:
    print("❌ Project root not found! Please ensure you have cloned the repo into your Drive.")
    # Fallback to default
    PROJECT_ROOT = "/content/drive/MyDrive/hope"
    os.makedirs(PROJECT_ROOT, exist_ok=True)

print(f"📂 Project Root: {PROJECT_ROOT}")

# 3. Add to System Path
SCRIPTS_PATH = os.path.join(PROJECT_ROOT, "scripts")
if SCRIPTS_PATH not in sys.path:
    sys.path.insert(0, SCRIPTS_PATH) # Insert at front to override any stale packages

os.chdir(PROJECT_ROOT)
print(f"✅ System path updated. Current dir: {os.getcwd()}")

In [ ]:
# @title ## 2. Repository Synchronization
REPO_URL = "https://github.com/planetazul3/hope.git" # @param {type:"string"}
BRANCH = "main" # @param {type:"string"}

def sync_repo():
    if not os.path.exists(os.path.join(PROJECT_ROOT, ".git")):
        print(f"Cloning {REPO_URL}...")
        subprocess.check_call(f"git clone --depth 1 {REPO_URL} {PROJECT_ROOT}", shell=True)
    else:
        print(f"Updating repository ({BRANCH})...")
        subprocess.check_call(f"git -C {PROJECT_ROOT} fetch origin", shell=True)
        subprocess.check_call(f"git -C {PROJECT_ROOT} reset --hard origin/{BRANCH}", shell=True)

try:
    sync_repo()
    print("✨ Repository is up to date.")
except Exception as e:
    print(f"⚠️ Sync failed (using existing files): {e}")

In [ ]:
# @title ## 3. Hardware & Dependency Audit (Transformer-Friendly)

print("📦 Installing modern dependency stack... (This takes a minute)")

# 1. Consolidate installations. Grouping them prevents pip from resolving 
# dependencies multiple times, which is significantly faster.
!pip install -q "numpy>=2.1,<2.3"
!pip install -q --index-url https://download.pytorch.org/whl/cu121 torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1
!pip install -q transformers==4.49.0 datasets==3.3.2 accelerate==1.4.0 tokenizers==0.21.0 sentencepiece==0.2.0 evaluate==0.4.3 pandas==2.2.3 scikit-learn==1.6.1 tqdm==4.67.1 matplotlib==3.9.3 seaborn==0.13.2 tensorboard==2.19.0 onnx==1.17.0 onnxruntime-gpu==1.20.1

# 2. Clean the UI. This wipes the massive wall of red/white pip logs.
from IPython.display import clear_output
clear_output()

import platform
import sys
import os

print("✅ Dependencies successfully installed.\n")
print("🔍 Environment Audit:")
print(f"💻 OS: {platform.platform()}")
print(f"🐍 Python: {platform.python_version()}")

# 3. GPU Audit via CLI (Avoids premature torch import)
try:
    # Gets exact GPU name, total VRAM, and current utilization in one clean line
    gpu_info = subprocess.check_output(
        "nvidia-smi --query-gpu=name,memory.total,utilization.gpu --format=csv,noheader", 
        shell=True
    ).decode().strip()
    print(f"🚀 GPU Active: {gpu_info}")
except Exception:
    print("❌ PyTorch/nvidia-smi does NOT detect a GPU. Training will be extremely slow.")

print("\n⚠️ CRITICAL STEP REQUIRED ⚠️")
print("Because core C-libraries (NumPy/PyTorch) were updated, you must restart the Python engine.")
print("Go to: Runtime ➡️ Restart runtime (or press Ctrl+M + .)")

In [ ]:
# @title ## 4. Credentials & Environment Editor
# @markdown Edit your `.env` file directly from here. Values are persisted to your Google Drive.

ENV_PATH = os.path.join(PROJECT_ROOT, ".env")
EXAMPLE_PATH = os.path.join(PROJECT_ROOT, ".env.example")

# Initialize if missing
if not os.path.exists(ENV_PATH) and os.path.exists(EXAMPLE_PATH):
    subprocess.check_call(f"cp {EXAMPLE_PATH} {ENV_PATH}", shell=True)

DERIV_DEMO_TOKEN = "" # @param {type:"string"}
DERIV_REAL_TOKEN = "" # @param {type:"string"}
MODEL_SIGNING_KEY = "" # @param {type:"string"}
DERIV_SYMBOL = "R_100" # @param {type:"string"}

def manage_env():
    updates = {}
    if DERIV_DEMO_TOKEN: updates["DERIV_DEMO_TOKEN"] = DERIV_DEMO_TOKEN
    if DERIV_REAL_TOKEN: updates["DERIV_REAL_TOKEN"] = DERIV_REAL_TOKEN
    if MODEL_SIGNING_KEY: updates["MODEL_SIGNING_KEY"] = MODEL_SIGNING_KEY
    if DERIV_SYMBOL: updates["DERIV_SYMBOL"] = DERIV_SYMBOL
    
    if os.path.exists(ENV_PATH):
        with open(ENV_PATH, "r") as f: lines = f.readlines()
        new_lines = []
        keys_seen = set()
        for line in lines:
            k = line.split("=")[0].strip() if "=" in line else None
            if k in updates:
                new_lines.append(f"{k}={updates[k]}\n")
                keys_seen.add(k)
            else:
                new_lines.append(line)
        for k, v in updates.items():
            if k not in keys_seen:
                new_lines.append(f"{k}={v}\n")
        with open(ENV_PATH, "w") as f: f.writelines(new_lines)
    
    # Load into current session environment
    if os.path.exists(ENV_PATH):
        with open(ENV_PATH, "r") as f:
            for line in f:
                if "=" in line and not line.startswith("#"):
                    k, v = line.strip().split("=", 1)
                    os.environ[k] = v.strip("\"\'")
    print("✅ Environment configuration synchronized.")

manage_env()

In [ ]:
# @title ## 5. Idempotent Tick Collection & Export

import os
import sys
import subprocess
import sqlite3
import pandas as pd

# ===================== CONFIGURATION =====================
PROJECT_ROOT = "/content/drive/MyDrive/hope"
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
ENV_PATH  = os.path.join(PROJECT_ROOT, ".env")
SCRIPTS   = os.path.join(PROJECT_ROOT, "scripts")

os.makedirs(DATA_DIR, exist_ok=True)

# --- Extract Symbol ---------------------------------------
SYMBOL = None
if os.path.exists(ENV_PATH):
    with open(ENV_PATH) as f:
        for line in f:
            line = line.strip()
            # Must start exactly with the key (ignores # comments)
            if line.startswith("DERIV_SYMBOL="):
                SYMBOL = line.split("=", 1)[1].strip('\'" ')
                break

if not SYMBOL:
    raise RuntimeError("❌ DERIV_SYMBOL not found in .env")
print(f"📌 Target Symbol: {SYMBOL}")

# --- Paths -------------------------------------------------
DB_PATH  = os.path.join(DATA_DIR, "tick_store.db")
CSV_PATH = os.path.join(DATA_DIR, f"{SYMBOL}_ticks.csv")

# --- Fast CSV Symbol Check ---------------------------------
if os.path.exists(CSV_PATH) and os.path.getsize(CSV_PATH) > 0:
    try:
        # Read only the first data row to avoid memory bloat
        first_row = pd.read_csv(CSV_PATH, header=None, nrows=1)
        if first_row.shape[1] >= 3:
            existing_sym = str(first_row.iloc[0, 0]).strip()
            if existing_sym not in (SYMBOL, "symbol"): 
                print(f"⚠️ Symbol changed (found {existing_sym}). Wiping {CSV_PATH}.")
                os.remove(CSV_PATH)
    except Exception as e:
        print(f"⚠️ Could not verify CSV symbol: {e}")

# ===================== COLLECTION ==========================
def run_command(cmd, desc):
    print(f"▶ {desc}\n   {' '.join(cmd)}")
    res = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
    if res.returncode != 0:
        print(f"❌ Failed:\n{res.stderr.strip()[-1000:]}")
        raise subprocess.CalledProcessError(res.returncode, cmd)
    print("   ✅ Done.")

# 1. Backfill Database
try:
    run_command([sys.executable, os.path.join(SCRIPTS, "tick_collector.py"), 
                 "--symbol", SYMBOL, "--db", DB_PATH, "--mode", "backfill"], 
                 "Fetching missing ticks")
except Exception:
    print("⚠️ Collector offline. Proceeding with local DB data.")

# 2. Export DB -> CSV
try:
    run_command([sys.executable, os.path.join(SCRIPTS, "export_db.py"), 
                 "--db", DB_PATH, "--csv", CSV_PATH, "--symbol", SYMBOL, 
                 "--incremental", "--validate"], 
                 "Exporting to CSV")
except Exception:
    print("⚠️ Incremental export failed. Using chunked SQLite fallback to prevent OOM...")
    conn = sqlite3.connect(DB_PATH)
    # Chunking prevents RAM exhaustion on HFT datasets
    chunks = pd.read_sql_query(f"SELECT symbol, epoch, quote FROM ticks WHERE symbol = '{SYMBOL}' ORDER BY epoch", conn, chunksize=500000)
    first_chunk = True
    for chunk in chunks:
        chunk.to_csv(CSV_PATH, mode='w' if first_chunk else 'a', index=False, header=False)
        first_chunk = False
    conn.close()
    print("   ✅ Fallback export complete.")

# ===================== LOAD IN MEMORY ======================
print("📊 Loading dataset into memory for analysis...")
# Load once here, Cell 6 will reuse it
df_ticks = pd.read_csv(CSV_PATH, header=None, names=["symbol", "epoch", "quote"], on_bad_lines='skip')

# Handle case where export didn't include symbol column
if pd.isna(pd.to_numeric(df_ticks['quote'], errors='coerce')).all():
    df_ticks = pd.read_csv(CSV_PATH, header=None, names=["epoch", "quote"], on_bad_lines='skip')

# Clean and enforce types
df_ticks['epoch'] = pd.to_numeric(df_ticks['epoch'], errors='coerce')
df_ticks['quote'] = pd.to_numeric(df_ticks['quote'], errors='coerce')
df_ticks.dropna(subset=['epoch', 'quote'], inplace=True)

if df_ticks.empty or (df_ticks["quote"] <= 0).any():
    raise RuntimeError("❌ CSV contains invalid or zero-price data.")

print(f"✅ Ready: {len(df_ticks):,} ticks loaded.")

In [ ]:
# @title ## 6. Data Integrity & Distribution Analysis

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Check if df_ticks is in memory from Cell 5. If not, fail gracefully.
if 'df_ticks' not in locals() or df_ticks.empty:
    raise RuntimeError("❌ df_ticks not found in memory. Run Cell 5 first.")

print(f"📈 Analyzing {len(df_ticks):,} ticks for {SYMBOL}...")

# Filter out duplicate timestamps (common in synthetic HFT streams)
df_clean = df_ticks.drop_duplicates(subset=['epoch']).copy()
dropped = len(df_ticks) - len(df_clean)
if dropped > 0:
    print(f"⚠️ Removed {dropped:,} duplicate epoch entries.")

sns.set_theme(style="darkgrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Recent Price Action
# Plotted against epoch rather than arbitrary index for true time scaling
tail_data = df_clean.tail(5000)
axes[0].plot(tail_data["epoch"], tail_data["quote"], color='blue', linewidth=1.2)
axes[0].set_title("Recent Price Action (Last 5k Ticks)", fontweight='bold')
axes[0].set_xlabel("Epoch Timestamp")
axes[0].set_ylabel("Quote Price")

# Plot 2: Volatility Distribution
# Calculate log returns (shift 1 tick)
returns = np.log(df_clean["quote"] / df_clean["quote"].shift(1)).dropna()

sns.histplot(returns, kde=True, bins=150, ax=axes[1], color='purple')
axes[1].set_title("Tick-to-Tick Volatility (Log Returns)", fontweight='bold')
axes[1].set_xlabel("Log Return")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

# Print basic distribution stats for modeling
print("\n📊 Return Distribution Stats:")
print(f"Mean: {returns.mean():.6f} | Std Dev: {returns.std():.6f}")
print(f"Skew: {returns.skew():.4f}  | Kurtosis: {returns.kurtosis():.4f}")

In [ ]:
# @title ## 7. Training & Live Monitoring (Production Aligned)

%load_ext tensorboard

import os
import sys
import subprocess
import traceback
import gc

# =============================================================================
# 1. Environment & Safe Paths
# =============================================================================
PROJECT_ROOT = "/content/drive/MyDrive/hope"
SCRIPTS_DIR  = os.path.join(PROJECT_ROOT, "scripts")
LOG_DIR      = os.path.join(PROJECT_ROOT, "logs")
ENV_PATH     = os.path.join(PROJECT_ROOT, ".env")

os.makedirs(LOG_DIR, exist_ok=True)

if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)

# CRITICAL: Change working directory so train_fixed.py saves artifacts 
# directly to the project root, as mandated by AGENTS.md
os.chdir(PROJECT_ROOT)

# =============================================================================
# 2. Hardened Symbol Extraction (Prioritize .env)
# =============================================================================
SYMBOL = None

# 1st Priority: Always read fresh from the .env file to bypass stale Colab cache
if os.path.exists(ENV_PATH):
    with open(ENV_PATH) as f:
        for line in f:
            line = line.strip()
            # Strict startswith prevents fetching commented-out symbols
            if line.startswith("DERIV_SYMBOL="):
                SYMBOL = line.split("=", 1)[1].strip('"\' ')
                break

# 2nd Priority: Fallback to environment variable
if not SYMBOL:
    SYMBOL = os.environ.get("DERIV_SYMBOL")

if not SYMBOL:
    raise RuntimeError("❌ DERIV_SYMBOL not found in .env. Set it and run Cell 5.")

# Force update the environment so downstream scripts (train_fixed.py) see the fresh value
os.environ["DERIV_SYMBOL"] = SYMBOL
print(f"📌 Training Target: {SYMBOL}")

CSV_PATH = os.path.join(PROJECT_ROOT, "data", f"{SYMBOL}_ticks.csv")
if not os.path.exists(CSV_PATH) or os.path.getsize(CSV_PATH) == 0:
    raise FileNotFoundError(f"❌ Tick data missing or empty at {CSV_PATH}. Run Cell 5.")
print(f"📂 Dataset verified: {CSV_PATH}")

# =============================================================================
# 3. Hardware Audit
# =============================================================================
import torch
if torch.cuda.is_available():
    print(f"🚀 Training on GPU: {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()
else:
    print("⚠️ WARNING: GPU NOT detected. Training will be extremely slow.")

# =============================================================================
# 4. Live Monitoring (TensorBoard)
# =============================================================================
print(f"📈 Launching TensorBoard logs → {LOG_DIR}")
os.system("pkill -f tensorboard")
%tensorboard --logdir {LOG_DIR}

# =============================================================================
# 5. Training Pipeline
# =============================================================================
print("\n🏗️ Starting ML pipeline (Phase 1: TS2Vec, Phase 2: Supervised MTL)")
print("   Architecture: Canonical Causal Transformer (L=32, 8 features)")

try:
    import train_fixed
    
    # Run the main training loop
    train_fixed.main(csv_path=CSV_PATH, log_dir=LOG_DIR)
    print("\n✅ Training finished successfully.")

    # Verify artifacts are correctly positioned for the Rust engine
    print("\n📦 Verifying model artifacts in Project Root...")
    artifacts = ["model.onnx", "model.onnx.sig", "best_model.pth"]
    saved_count = 0
    
    for fname in artifacts:
        fpath = os.path.join(PROJECT_ROOT, fname)
        if os.path.exists(fpath):
            size_mb = os.path.getsize(fpath) / 1e6
            print(f"   • Verified: {fname} ({size_mb:.2f} MB)")
            saved_count += 1
            
    if saved_count == 0:
        print("   ⚠️ No output models found in project root. Did train_fixed fail to export?")
    elif saved_count < len(artifacts):
        print("   ⚠️ Some artifacts are missing. Check logs for quantization or signing errors.")

except ModuleNotFoundError:
    print("❌ Module `train_fixed` not found in `scripts/`.")
except Exception:
    print("❌ Training failed. Traceback:")
    traceback.print_exc()
finally:
    # Aggressive memory cleanup prevents OOM crashes when re-running this cell
    if 'train_fixed' in sys.modules:
        del sys.modules['train_fixed']
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()